# House Price Prediction - Exploratory Data Analysis & Modeling
**Course / Portfolio Project:** House Price Prediction Using Machine Learning  
**Dataset:** Ames Housing Dataset (1,460 residential properties, Ames, Iowa)  
**Objective:** Build, evaluate, and compare four regression models to predict residential house sale prices.

## 1. Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

## 2. Load the Dataset
Load the Ames housing dataset and inspect its structure, data types, and initial records.

In [ ]:
df = pd.read_csv('../data/house_prices.csv')
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 3. Data Preprocessing & Cleaning
1. Map `HouseStyle` to numeric `Floors`
2. Filter out extreme atypical non-arm length outliers (`GrLivArea > 4000` & `SalePrice < 300,000`)
3. Inspect missing values across selected features

In [ ]:
style_to_floors = {
    '1Story': 1.0, '1.5Fin': 1.5, '1.5Unf': 1.5,
    '2Story': 2.0, '2.5Fin': 2.5, '2.5Unf': 2.5,
    'SFoyer': 1.0, 'SLvl': 1.5
}
df['Floors'] = df['HouseStyle'].map(style_to_floors).fillna(1.0)

# Remove extreme outliers
initial_len = len(df)
df = df.drop(df[(df['GrLivArea'] > 4000) & (df['SalePrice'] < 300000)].index)
print(f"Removed {initial_len - len(df)} outlier(s). Cleaned shape: {df.shape}")

## 4. Exploratory Data Analysis (EDA)
Visualizing target distribution, correlations, living area, bedrooms, and neighborhood averages.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df['SalePrice'], kde=True, color='#2563eb', bins=40, ax=ax)
ax.set_title('Distribution of House Sale Prices')
ax.set_xlabel('Sale Price ($)')
plt.show()

In [ ]:
numeric_features = ['OverallQual', 'GrLivArea', 'BedroomAbvGr', 'FullBath', 'Floors', 'GarageCars', 'YearBuilt', 'TotalBsmtSF', 'LotArea', 'OverallCond']
plt.figure(figsize=(10, 8))
sns.heatmap(df[numeric_features + ['SalePrice']].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

## 5. Model Training & Comparison
We split the dataset 80:20 into training and testing sets, build a leakage-free `ColumnTransformer` pipeline, and train:
1. Linear Regression
2. Decision Tree Regressor
3. Random Forest Regressor
4. Gradient Boosting Regressor

In [ ]:
categorical_features = ['Neighborhood']
X = df[numeric_features + categorical_features]
y = df['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_features)
])

models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=150, max_depth=16, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=150, learning_rate=0.08, max_depth=4, random_state=42)
}

for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('reg', model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    print(f"{name:20s} | Test R2: {r2:.4f} | MAE: ${mae:,.0f} | RMSE: ${rmse:,.0f}")